# Parte 2 · La geometría de una predicción (2.1 y 2.2)

**Autor:** Herman Echeverría · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

Datos: Taiwan Credit, estandarizados como en 0.1, con los hiperparámetros de la validación cruzada (`resultados/hiperparametros.json`). La ficha de evidencia de cada predicción se resume con

$$N_{\text{ef}}(x) = \frac{\left(\sum_i w_i(x)\right)^2}{\sum_i w_i(x)^2}, \qquad c(x) = \frac{\max_i w_i(x)^2}{\sum_j w_j(x)^2},$$

calculados con `src/evidencia.py`.

In [ ]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from datos import cargar_taiwan
from pesos import pesos_ols, pesos_ridge, pesos_knn, pesos_arbol
from bosque import pesos_bosque, verificar
from evidencia import n_efectivo, concentracion, testigos

X_train, X_test, y_train, y_test, sd = cargar_taiwan("../data/taiwan_credit.csv")
X_train_crudo, X_test_crudo, _, _, _ = cargar_taiwan("../data/taiwan_credit.csv", escalar=False)  # para leer los testigos
n = len(X_train)

hp = json.load(open("../resultados/hiperparametros.json"))
hp

In [ ]:
modelos = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=hp["lambda"]),
    "k-NN": KNeighborsRegressor(n_neighbors=hp["k"]),
    "Árbol": DecisionTreeRegressor(max_depth=hp["max_depth"], random_state=0),
    "Bosque": RandomForestRegressor(n_estimators=200, min_samples_leaf=hp["min_samples_leaf"],
                                    max_features=0.33, bootstrap=False, random_state=0, n_jobs=-1),
}
for m in modelos.values():
    m.fit(X_train, y_train)


def pesos(nombre, X_q):
    if nombre == "OLS":
        return pesos_ols(X_train, X_q)
    if nombre == "Ridge":
        return pesos_ridge(X_train, X_q, hp["lambda"])
    if nombre == "k-NN":
        return pesos_knn(modelos["k-NN"], X_q, n)
    if nombre == "Árbol":
        return pesos_arbol(modelos["Árbol"], X_train, X_q)
    return pesos_bosque(modelos["Bosque"], X_train, X_q)

## 2.1 · $N_{\text{ef}}$ del bosque en todo el conjunto de prueba

La matriz completa de pesos serían $6\,000 \times 24\,000$ números (~1.2 GB), así que se calcula por lotes de 500 consultas y de cada lote solo se guardan $N_{\text{ef}}$ y $c(x)$. En cada lote se verifica además que los pesos reproducen el `.predict()`.

In [ ]:
lote = 500
nef_bosque, c_bosque = np.empty(len(X_test)), np.empty(len(X_test))
for inicio in range(0, len(X_test), lote):
    X_q = X_test.iloc[inicio:inicio + lote]
    W = pesos_bosque(modelos["Bosque"], X_train, X_q)
    verificar(W, y_train, modelos["Bosque"].predict(X_q))
    nef_bosque[inicio:inicio + lote] = n_efectivo(W)
    c_bosque[inicio:inicio + lote] = concentracion(W)

pd.DataFrame({"N_ef del bosque": nef_bosque, "c(x) del bosque": c_bosque}).describe().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(nef_bosque, bins=60)
ax.axvline(np.median(nef_bosque), color="tab:red", ls="--", label=f"mediana = {np.median(nef_bosque):.0f}")
ax.set_xlabel(r"$N_{ef}(x)$ del bosque"); ax.set_ylabel("filas de prueba")
ax.set_title("¿En cuántas filas se apoya cada predicción del bosque?")
ax.legend(); fig.tight_layout()
fig.savefig("../figs/parte2_nef_bosque.png", dpi=150)
plt.show()